
## Folder 03 — Bronze Ingestion with Auto Loader


**Ingest all seven CSV datasets from the landing directory into Bronze Delta tables using Databricks Auto Loader.**


```text
┌──────────────────────────────────────────────┐
│                 LANDING LAYER                │
│                                              │
│              7 Original CSV Files            │
│                                              │
│  customers.csv       products.csv            │
│  orders.csv          order_items.csv         │
│  payments.csv        shipments.csv           │
│  inventory_snapshots.csv                     │
└──────────────────────────────────────────────┘
                       │
                       ▼
┌──────────────────────────────────────────────┐
│                  AUTO LOADER                  │
│                                              │
│  • Schema Inference                          │
│  • Incremental File Discovery                │
│  • Checkpoint Management                     │
└──────────────────────────────────────────────┘
                       │
                       ▼
┌──────────────────────────────────────────────┐
│             BRONZE LAYER - DELTA             │
│                                              │
│  • Raw Source Records                        │
│  • Ingestion Metadata                        │
│  • Source File Information                   │
│  • Ingestion Timestamp                       │
└──────────────────────────────────────────────┘
                       │
                       ▼
┌──────────────────────────────────────────────┐
│              BRONZE DELTA TABLES             │
│                                              │
│  1. bronze.customers                         │
│  2. bronze.products                          │
│  3. bronze.orders                            │
│  4. bronze.order_items                       │
│  5. bronze.payments                          │
│  6. bronze.shipments                         │
│  7. bronze.inventory_snapshots               │
└──────────────────────────────────────────────┘
```

In [0]:
# ------------------------------------------------------------
# 1. IMPORTS
# ------------------------------------------------------------

from pyspark.sql import functions as F


# ------------------------------------------------------------
# 2. CONFIGURATION
# ------------------------------------------------------------

CATALOG = "workspace"

BASE_PATH = (
    "/Volumes/workspace/default/data_volume/"
    "ecommerce_lakehouse"
)

LANDING_PATH = f"{BASE_PATH}/landing"
BRONZE_PATH = f"{BASE_PATH}/bronze"

CHECKPOINT_PATH = f"{BASE_PATH}/checkpoints/bronze"
SCHEMA_PATH = f"{BASE_PATH}/schemas/bronze"

BRONZE_SCHEMA = "ecommerce_bronze"

datasets = [
    "customers",
    "products",
    "orders",
    "order_items",
    "payments",
    "shipments",
    "inventory_snapshots"
]


# ------------------------------------------------------------
# 3. CREATE BRONZE SCHEMA
# ------------------------------------------------------------

spark.sql(
    f"""
    CREATE SCHEMA IF NOT EXISTS
    {CATALOG}.{BRONZE_SCHEMA}
    """
)

print("Bronze schema ready.")
print(f"Schema: {CATALOG}.{BRONZE_SCHEMA}")


# ------------------------------------------------------------
# 4. BRONZE INGESTION FUNCTION
# ------------------------------------------------------------

def ingest_bronze(dataset):

    source_path = f"{LANDING_PATH}/{dataset}"
    target_path = f"{BRONZE_PATH}/{dataset}"

    checkpoint_path = f"{CHECKPOINT_PATH}/{dataset}"
    schema_path = f"{SCHEMA_PATH}/{dataset}"

    print("\n" + "=" * 70)
    print(f"BRONZE INGESTION: {dataset.upper()}")
    print("=" * 70)

    print(f"Source      : {source_path}")
    print(f"Target      : {target_path}")
    print(f"Checkpoint  : {checkpoint_path}")
    print(f"Schema      : {schema_path}")


    # --------------------------------------------------------
    # 4.1 VALIDATE SOURCE FILES
    # --------------------------------------------------------

    source_files = [
        file
        for file in dbutils.fs.ls(source_path)
        if file.path.lower().endswith(".csv")
    ]

    if not source_files:
        raise ValueError(
            f"No CSV files found: {source_path}"
        )

    print(f"CSV files found: {len(source_files)}")


    # --------------------------------------------------------
    # 4.2 READ SOURCE USING AUTO LOADER
    # --------------------------------------------------------

    source_df = (

        spark.readStream

        .format("cloudFiles")

        .option("cloudFiles.format", "csv")

        .option(
            "cloudFiles.schemaLocation",
            schema_path
        )

        .option(
            "cloudFiles.inferColumnTypes",
            "true"
        )

        .option(
            "cloudFiles.includeExistingFiles",
            "true"
        )

        .option("header", "true")

        .option("mode", "PERMISSIVE")

        .load(source_path)

    )


    # --------------------------------------------------------
    # 4.3 ADD INGESTION METADATA
    # --------------------------------------------------------

    bronze_df = (

        source_df

        .withColumn(
            "_ingested_at",
            F.current_timestamp()
        )

        .withColumn(
            "_source_file",
            F.col("_metadata.file_path")
        )

    )


    # --------------------------------------------------------
    # 4.4 WRITE DELTA DATA TO VOLUME
    # --------------------------------------------------------

    query = (

        bronze_df

        .writeStream

        .format("delta")

        .outputMode("append")

        .option(
            "checkpointLocation",
            checkpoint_path
        )

        .option(
            "mergeSchema",
            "true"
        )

        .trigger(
            availableNow=True
        )

        .start(target_path)

    )

    query.awaitTermination()


    # --------------------------------------------------------
    # 4.5 VERIFY DELTA DATA
    # --------------------------------------------------------

    result_df = (

        spark.read

        .format("delta")

        .load(target_path)

    )

    record_count = result_df.count()

    if record_count == 0:
        raise ValueError(
            f"No records found in Bronze Delta path: {target_path}"
        )


    # --------------------------------------------------------
    # 4.6 VALIDATE METADATA COLUMNS
    # --------------------------------------------------------

    required_columns = [
        "_ingested_at",
        "_source_file"
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in result_df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing columns: {missing_columns}"
        )


    # --------------------------------------------------------
    # 4.7 DISPLAY DATA
    # --------------------------------------------------------

    print("\nINGESTION SUCCESSFUL")

    print(f"Dataset       : {dataset}")
    print(f"Record Count  : {record_count}")
    print(f"Delta Location: {target_path}")

    display(result_df.limit(5))


    return {

        "dataset": dataset,
        "records": record_count,
        "location": target_path,
        "status": "SUCCESS",
        "error": ""

    }


# ------------------------------------------------------------
# 5. PROCESS ALL DATASETS
# ------------------------------------------------------------

ingestion_results = []

for dataset in datasets:

    try:

        result = ingest_bronze(dataset)

        ingestion_results.append(result)


    except Exception as e:

        error_message = str(e)[:2000]

        print("\n" + "!" * 70)
        print(f"FAILED DATASET: {dataset}")
        print(error_message)
        print("!" * 70)

        ingestion_results.append({

            "dataset": dataset,
            "records": 0,
            "location": f"{BRONZE_PATH}/{dataset}",
            "status": "FAILED",
            "error": error_message

        })


# ------------------------------------------------------------
# 6. INGESTION SUMMARY
# ------------------------------------------------------------

summary_df = spark.createDataFrame(
    ingestion_results
)

display(summary_df)


# ------------------------------------------------------------
# 7. EXECUTION STATISTICS
# ------------------------------------------------------------

success_count = sum(
    1 for row in ingestion_results
    if row["status"] == "SUCCESS"
)

failed_count = sum(
    1 for row in ingestion_results
    if row["status"] == "FAILED"
)

print("\n" + "=" * 70)
print("BRONZE INGESTION SUMMARY")
print("=" * 70)

print(f"Total datasets : {len(datasets)}")
print(f"Successful     : {success_count}")
print(f"Failed         : {failed_count}")


# ------------------------------------------------------------
# 8. FINAL VALIDATION
# ------------------------------------------------------------

if failed_count > 0:

    failed_datasets = [
        row["dataset"]
        for row in ingestion_results
        if row["status"] == "FAILED"
    ]

    raise RuntimeError(
        f"Bronze ingestion failed: {failed_datasets}"
    )


print("\n" + "=" * 70)
print("ALL BRONZE DATASETS PROCESSED SUCCESSFULLY")
print("Delta files stored in Unity Catalog Volume.")
print("=" * 70)



## Verify the Bronze data


**After running the corrected notebook, execute:**

In [0]:
# Verify customers Bronze Delta data

CUSTOMERS_BRONZE_PATH = (
    "/Volumes/workspace/default/data_volume/"
    "ecommerce_lakehouse/bronze/customers"
)

df = (
    spark.read
    .format("delta")
    .load(CUSTOMERS_BRONZE_PATH)
)

print("Total customer records:", df.count())

display(df.limit(10))


**You can also inspect the physical files:**

In [0]:
datasets = [
    "customers",
    "products",
    "orders",
    "order_items",
    "payments",
    "shipments",
    "inventory_snapshots"
]

BRONZE_PATH = (
    "/Volumes/workspace/default/data_volume/"
    "ecommerce_lakehouse/bronze"
)

for dataset in datasets:

    path = f"{BRONZE_PATH}/{dataset}"

    print(f"\nDataset: {dataset}")

    try:
        df = spark.read.format("delta").load(path)

        print("Records:", df.count())
        print("Columns:", df.columns)

    except Exception as e:
        print("ERROR:", str(e)[:500])


## Important Interview Concepts:


| Concept | Explanation |
|---|---|
| **Auto Loader** | Incrementally discovers and processes new files. |
| **Schema Inference** | Automatically identifies source column names and data types. |
| **Schema Evolution** | Handles changes in incoming file schemas, such as adding new columns. |
| **Checkpointing** | Tracks processed files and streaming progress to support recovery. |
| **Delta Lake** | Stores Bronze data in transactional Delta tables with ACID properties. |
| **`availableNow=True`** | Processes all available data and stops automatically when caught up. |
